In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: read the raw CSV, then clean, engineer text, split, encode, and save that split.
# False: skip those steps and load the saved train, validation, and test frames from data/processed/split.
REBUILD = False


# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/07 21:54:38 WARN Utils: Your hostname, NineQueendeMacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.12.5.168 instead (on interface en0)
26/11/07 21:54:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/07 21:54:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


In [3]:
from src.embeddings import EmbeddingModel
# Fit the selected embedding before prediction; downstream models read `embedding`.
embedding_model = EmbeddingModel(df_train, df_valid, df_test)
embedding_name = "glove"

print(f"Embedding Model `{embedding_name}` Training")
EmbeddingModel.train_registry[embedding_name](embedding_model)


Embedding Model `glove` Training


BUILDING VOCABULARY
Processed 0 tokens.100000 tokens.200000 tokens.300000 tokens.400000 tokens.500000 tokens.600000 tokens.700000 tokens.800000 tokens.900000 tokens.1000000 tokens.1100000 tokens.1200000 tokens.1300000 tokens.1400000 tokens.1500000 tokens.1600000 tokens.1700000 tokens.1800000 tokens.1900000 tokens.2000000 tokens.2100000 tokens.2200000 tokens.2300000 tokens.2400000 tokens.2500000 tokens.2600000 tokens.2700000 tokens.2800000 tokens.2900000 tokens.3000000 tokens.3100000 tokens.3200000 tokens.3300000 tokens.3400000 tokens.3500000 tokens.3600000 tokens.3700000 tokens.3800000 tokens.3900000 tokens.4000000 tokens.4100000 tokens.4200000 tokens.4300000 tokens.4400000 tokens.4500000 tokens.4600000 tokens.4700000 tokens.4800000 tokens.4900000 tokens.5000000 tokens.5100000 tokens.5200000 tokens.5300000 tokens.5400000 tokens.5500000 tokens.5600000 tokens.5700000 tokens.5800000 tokens.5900000 tokens.6000000 tokens.6100000 tokens.6200000 tokens.6300000 tokens.6400000 tokens.6500000 to

20000002100000220000023000002400000250000026000002700000280000029000003000000310000032000003300000340000035000003600000370000038000003900000400000041000004200000430000044000004500000460000047000004800000490000050000005100000520000053000005400000550000056000005700000580000059000006000000610000062000006300000640000065000006600000670000068000006900000700000071000007200000730000074000007500000760000077000007800000790000080000008100000820000083000008400000850000086000008700000880000089000009000000910000092000009300000940000095000009600000970000098000009900000100000001010000010200000103000001040000010500000106000001070000010800000109000001100000011100000112000001130000011400000115000001160000011700000Processed 11714604 tokens.
Writing cooccurrences to disk........2 files in total.
Merging cooccurrence files: processed 0 lines.100000 lines.200000 lines.300000 lines.400000 lines.500000 lines.600000 lines.700000 lines.800000 lines.900000 lines.1000000 lines.1100000 lines.1200000 lines.1300000 l

Shuffling cooccurrences...


processed 15962656 lines.
Wrote 1 temporary file(s).
Merging temp files: processed 0 lines.15962656 lines.Merging temp files: processed 15962656 lines.

TRAINING MODEL
Read 15962656 lines.
Initializing parameters...Using random seed 1794059830
done.
vector size: 100
vocab size: 29506
x_max: 100.000000
alpha: 0.750000


Training model...


11/07/26 - 09:57.11PM, iter: 001, cost: 0.032413
11/07/26 - 09:57.12PM, iter: 002, cost: 0.022795
11/07/26 - 09:57.13PM, iter: 003, cost: 0.018890
11/07/26 - 09:57.15PM, iter: 004, cost: 0.016161
11/07/26 - 09:57.16PM, iter: 005, cost: 0.014158
11/07/26 - 09:57.17PM, iter: 006, cost: 0.012626
11/07/26 - 09:57.18PM, iter: 007, cost: 0.011445
11/07/26 - 09:57.20PM, iter: 008, cost: 0.010539
11/07/26 - 09:57.21PM, iter: 009, cost: 0.009846
11/07/26 - 09:57.22PM, iter: 010, cost: 0.009316
11/07/26 - 09:57.23PM, iter: 011, cost: 0.008911
11/07/26 - 09:57.25PM, iter: 012, cost: 0.008586
11/07/26 - 09:57.26PM, iter: 013, cost: 0.008329
11/07/26 - 09:57.27PM, iter: 014, cost: 0.008115
11/07/26 - 09:57.28PM, iter: 015, cost: 0.007940
11/07/26 - 09:57.30PM, iter: 016, cost: 0.007794
11/07/26 - 09:57.31PM, iter: 017, cost: 0.007662
11/07/26 - 09:57.32PM, iter: 018, cost: 0.007554
11/07/26 - 09:57.34PM, iter: 019, cost: 0.007457
11/07/26 - 09:57.35PM, iter: 020, cost: 0.007369
11/07/26 - 09:57.36P

In [4]:
train_embedding_dict = {}
valid_embedding_dict = {}
test_embedding_dict = {}

print(f"Embedding Model `{embedding_name}` Running")
train_embedding_dict[embedding_name], valid_embedding_dict[embedding_name], test_embedding_dict[embedding_name] = EmbeddingModel.predict_registry[embedding_name](embedding_model)
data_oi.save_embedding_data(
    embedding_name,
    train_embedding_dict[embedding_name],
    valid_embedding_dict[embedding_name],
    test_embedding_dict[embedding_name],
)


Embedding Model `glove` Running


In [5]:
from src.analytics import AnalyticsModel

analytics_models = {}
analytics_models[embedding_name] = AnalyticsModel(
    embedding_name,
    train_embedding_dict[embedding_name],
    valid_embedding_dict[embedding_name],
    test_embedding_dict[embedding_name],
)


In [6]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}
for name, function in AnalyticsModel.registry.items():
    print(f"Embedding Model `{embedding_name}` Analytics Model `{name}` Running")
    # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
    model_name = f"{name}__{embedding_name}"
    train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = function(
        analytics_models[embedding_name]
    )
    data_oi.save_prediction_data(
        model_name,
        train_pred_dict[model_name],
        valid_pred_dict[model_name],
        test_pred_dict[model_name],
    )


Embedding Model `glove` Analytics Model `lr_improved` Running


In [7]:
from src.evaluation import Scorer
train_pred_dict, valid_pred_dict , test_pred_dict = data_oi.load_prediction_data()
# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)

Training Data:
+---------------+--------------+----------+------------------+-----------+----------+----------+
|Embedding_Model|Analytic_Model|  accuracy|weighted_precision|weighted_f1|       mae|      rmse|
+---------------+--------------+----------+------------------+-----------+----------+----------+
|          tfidf|   lr_improved|0.75012853|        0.75981325| 0.74796270|0.69973008|1.80319981|
|       baseline|            lr|0.99976864|        0.99976864| 0.99976863|0.00048843|0.04967762|
|          glove|   lr_improved|0.36547558|        0.41426571| 0.38324315|1.89402314|2.98147882|
+---------------+--------------+----------+------------------+-----------+----------+----------+

Validation Data:
+---------------+--------------+----------+------------------+-----------+----------+----------+
|Embedding_Model|Analytic_Model|  accuracy|weighted_precision|weighted_f1|       mae|      rmse|
+---------------+--------------+----------+------------------+-----------+----------+---------